<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.
Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:

using System;
using System.Collections.Generic;

// Интерфейс
public interface IPaymentMethod
{
    int PaymentMethodId { get; }
    string MethodName { get; }
    decimal MinAmount { get; }
    bool CheckMinimumAmount(decimal amount);
    string ProcessPayment(decimal amount);
    string GetPaymentDetails();
}


public abstract class PaymentMethod : IPaymentMethod
{
    public int PaymentMethodId { get; }
    public string MethodName { get; }
    public decimal MinAmount { get; }

    protected PaymentMethod(int id, string methodName, decimal minAmount)
    {
        PaymentMethodId = id;
        MethodName = methodName;
        MinAmount = minAmount;
    }

    public virtual bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;

    public virtual string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return $"Ошибка: сумма {amount:C} меньше минимальной ({MinAmount:C}).";
        return $"Оплата {amount:C} через '{MethodName}' выполнена.";
    }

    public virtual string GetPaymentDetails()
        => $"ID: {PaymentMethodId}, Метод: {MethodName}, Мин. сумма: {MinAmount:C}";
}

//Онлайн-оплата 
public class OnlinePayment : PaymentMethod
{
    public string PaymentUrl { get; }

    public OnlinePayment(int id, string methodName, decimal minAmount, string paymentUrl)
        : base(id, methodName, minAmount) { PaymentUrl = paymentUrl; }

    public override string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return $"Ошибка: сумма {amount:C} меньше минимальной ({MinAmount:C}).";
        return $"Онлайн-оплата {amount:C} через '{MethodName}'. Перейдите: {PaymentUrl}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $", URL: {PaymentUrl}";
}

//Банковский перевод
public class BankTransfer : PaymentMethod
{
    public string BankData { get; }
    public decimal CommissionPercent { get; }

    public BankTransfer(int id, string methodName, decimal minAmount,
                        string bankData, decimal commissionPercent = 1.5m)
        : base(id, methodName, minAmount)
    {
        BankData = bankData;
        CommissionPercent = commissionPercent;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount * CommissionPercent / 100m;
        return (amount + commission) >= MinAmount;
    }

    public override string ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return $"Ошибка: сумма {amount:C} с комиссией {CommissionPercent}% меньше минимальной ({MinAmount:C}).";
        decimal commission = amount * CommissionPercent / 100m;
        return $"Банковский перевод {amount:C} (комиссия {commission:C}). Реквизиты: {BankData}";
    }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $", Банк. данные: {BankData}, Комиссия: {CommissionPercent}%";
}

// Наличные 
public class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; }

    public CashPayment(int id, string methodName, decimal minAmount, string cashPickupPoint)
        : base(id, methodName, minAmount) { CashPickupPoint = cashPickupPoint; }

    public override string GetPaymentDetails()
        => base.GetPaymentDetails() + $", Место выдачи наличных: {CashPickupPoint}";
}

// Запуск 
public static class PaymentDemo
{
    public static void Run()
    {
        List<IPaymentMethod> methods = new()
        {
            new OnlinePayment(1, "Онлайн-оплата картой", 100m, "https://pay.example.com/checkout"),
            new BankTransfer(2, "Банковский перевод", 1000m, "IBAN: DE12 3456 7890", 2.0m),
            new CashPayment(3, "Наличные", 50m, "г. Москва, ул. Ленина, 10")
        };

        Console.WriteLine("=== Способы оплаты ===\n");
        foreach (var m in methods)
            Console.WriteLine(m.GetPaymentDetails());

        Console.WriteLine("\n=== Обработка платежей ===\n");
        decimal[] amounts = { 50m, 500m, 2000m };

        foreach (var m in methods)
        {
            Console.WriteLine($"--- {m.MethodName} ---");
            foreach (var amount in amounts)
                Console.WriteLine($"  {amount:C} -> {m.ProcessPayment(amount)}");
            Console.WriteLine();
        }
    }
}

PaymentDemo.Run();

=== Способы оплаты ===

ID: 1, Метод: Онлайн-оплата картой, Мин. сумма: ¤100.00, URL: https://pay.example.com/checkout
ID: 2, Метод: Банковский перевод, Мин. сумма: ¤1,000.00, Банк. данные: IBAN: DE12 3456 7890, Комиссия: 2.0%
ID: 3, Метод: Наличные, Мин. сумма: ¤50.00, Место выдачи наличных: г. Москва, ул. Ленина, 10

=== Обработка платежей ===

--- Онлайн-оплата картой ---
  ¤50.00 -> Ошибка: сумма ¤50.00 меньше минимальной (¤100.00).
  ¤500.00 -> Онлайн-оплата ¤500.00 через 'Онлайн-оплата картой'. Перейдите: https://pay.example.com/checkout
  ¤2,000.00 -> Онлайн-оплата ¤2,000.00 через 'Онлайн-оплата картой'. Перейдите: https://pay.example.com/checkout

--- Банковский перевод ---
  ¤50.00 -> Ошибка: сумма ¤50.00 с комиссией 2.0% меньше минимальной (¤1,000.00).
  ¤500.00 -> Ошибка: сумма ¤500.00 с комиссией 2.0% меньше минимальной (¤1,000.00).
  ¤2,000.00 -> Банковский перевод ¤2,000.00 (комиссия ¤40.00). Реквизиты: IBAN: DE12 3456 7890

--- Наличные ---
  ¤50.00 -> Оплата ¤50.00 чере